# ARC-AGI-2 SFT in Colab -> private Kaggle Model `qwen3-4b-grids15-sft-a`

One notebook = one training run = one private Kaggle Model.

1. Runtime > Change runtime type > GPU (T4 free tier works; L4/A100 finish sooner).
2. Add a Colab secret named `KAGGLE_CREDENTIALS` holding the contents of
   `~/.kaggle/credentials.json` (the OAuth pair, not the legacy key).
3. Runtime > Run all.

Everything is resumable: the checkpoint and generated data live on Google Drive, so after a
disconnect `Run all` picks up at the last saved checkpoint (`--resume`). Do not edit cells
while a run is in flight. Details and per-cell pass lines: `arc-agi-2/dev/colab_sft_runbook.md`.


In [ ]:
import subprocess
r = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                   capture_output=True, text=True)
print(r.stdout.strip() or r.stderr.strip())
name = r.stdout.lower()
PRESET = "a100" if "a100" in name else "l4" if "l4" in name else "t4" if "t4" in name else None
assert PRESET, "no GPU on this runtime -- Runtime > Change runtime type > GPU"
print("preset:", PRESET)


In [ ]:
import subprocess
subprocess.run("pip -q install 'transformers>=4.55,<5' 'peft>=0.13' safetensors 'kaggle>=2'",
               shell=True, check=True)
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
assert torch.cuda.is_available()


In [ ]:
import json, os, subprocess
from google.colab import userdata, drive

os.makedirs("/root/.kaggle", exist_ok=True)
cred = userdata.get("KAGGLE_CREDENTIALS")
assert cred, "add the Colab secret KAGGLE_CREDENTIALS = contents of ~/.kaggle/credentials.json"
open("/root/.kaggle/credentials.json", "w").write(cred.strip())
os.chmod("/root/.kaggle/credentials.json", 0o600)
OWNER = json.loads(cred).get("username") or "takumuhata"
RUN_NAME = "qwen3-4b-grids15-sft-a"
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/arc_sft/" + RUN_NAME
os.makedirs(OUT, exist_ok=True)
r = subprocess.run(["kaggle", "models", "list", "-m"], capture_output=True, text=True)
print(r.stdout[:400] or r.stderr[:400])
print("owner:", OWNER, "| out:", OUT)


In [ ]:
# unpack the embedded repo subset (6 scripts + 4 public ARC-AGI-2 JSONs)
import base64, glob, io, tarfile, zlib
BLOB = (
    "..."
    "P+fn/Jyf8/OGz/8HTCspCA=="
)
raw = zlib.decompress(base64.b64decode(BLOB))
with tarfile.open(fileobj=io.BytesIO(raw)) as tf:
    tf.extractall("/content/repo", filter="data")
print(len(glob.glob("/content/repo/**/*.*", recursive=True)), "files under /content/repo")


In [ ]:
# base checkpoint + environment self-checks (~3 min CPU for the trainer selftest)
import os, subprocess
if not os.path.exists("/content/base/config.json"):
    subprocess.run("kaggle models instances versions download "
                   "sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1 -p /content/base --untar", shell=True, check=True)
subprocess.run("cd /content/repo && python arc-agi-2/dev/check_swap_model.py /content/base /content/base",
               shell=True, check=True)
subprocess.run("cd /content/repo && python arc-agi-2/dev/check_sft_data.py --selftest", shell=True, check=True)
subprocess.run("cd /content/repo && python arc-agi-2/dev/sft_ag2.py --selftest", shell=True, check=True)


In [ ]:
# data: synthetic tasks + the Nabidnur 410, then the contamination/format check.
# Files live on Drive: regenerated deterministically (seed 0) only if missing.
import os, subprocess
SYNTH = OUT + "/synth.jsonl"
NAB = OUT + "/nabidnur.jsonl"
DATA = ['tasks:arc-agi-2/arc-agi_training_challenges.json:arc-agi-2/arc-agi_training_solutions.json@0.65', "jsonl:" + SYNTH + "@0.3", "jsonl:" + NAB + "@0.05"]
if not os.path.exists(NAB):
    subprocess.run("wget -q -O " + NAB + " https://huggingface.co/datasets/Nabidnur/arc-agi-2-grids/resolve/main/synthetic/curriculum_v0_verified.jsonl", shell=True, check=True)
if not os.path.exists(SYNTH):
    subprocess.run(f"cd /content/repo && python arc-agi-2/dev/gen_synth_ag2.py --n 3000 "
                   f"--out {SYNTH} --seed 0", shell=True, check=True)
subprocess.run("cd /content/repo && python arc-agi-2/dev/gen_synth_ag2.py --verify " + SYNTH,
               shell=True, check=True)
subprocess.run("cd /content/repo && python arc-agi-2/dev/check_sft_data.py " +
               " ".join(s.split("@")[0] for s in DATA) + " --write-exclude " + OUT + "/exclude.json",
               shell=True, check=True)


In [ ]:
# train (auto-resumes from the Drive checkpoint after a disconnect)
import os, subprocess
cmd = ("cd /content/repo && python arc-agi-2/dev/sft_ag2.py --model /content/base --out " + OUT +
       " --preset " + PRESET + " --max-steps 4000 --max-hours 10.5"
       " --exclude " + OUT + "/exclude.json --val-from first" + "".join(" --data " + s for s in DATA))
if os.path.exists(OUT + "/ckpt.pt"):
    cmd += " --resume"
    print("resuming from the Drive checkpoint")
print(cmd)
subprocess.run(cmd, shell=True, check=True)


In [ ]:
# gate: nothing leaves this notebook unless the merged model is a drop-in AND val improved
import json
m = json.load(open(OUT + "/manifest.json"))
print(json.dumps({{k: m[k] for k in ("steps", "updates", "train_loss_first", "train_loss_last",
                                    "val_loss_before", "val_loss_after", "val_improved",
                                    "drop_in_ok", "fp16_guard")}}, indent=1))
assert m["drop_in_ok"], "merged dir is not a drop-in for the base -- do not upload"
assert m["val_improved"], "held-out val loss got worse -- do not upload (a panel would waste a run)"
print("GATE PASS -- uploading as a private Kaggle Model")


In [ ]:
# upload: model resource, then the merged dir as a transformers/bf16 instance.
# UNVERIFIED on Colab: if this fails, merged weights stay on Drive -- upload from the ops VM
# instead (colab_sft_runbook.md, "hand over").
import json, os, subprocess
MERGED = OUT + "/merged"
meta_dir = "/content/meta_" + RUN_NAME
os.makedirs(meta_dir, exist_ok=True)
subprocess.run("kaggle models init -p " + meta_dir, shell=True, check=True)
mp = meta_dir + "/model-metadata.json"
meta = json.load(open(mp))
meta.update(ownerSlug=OWNER, title=RUN_NAME, slug=RUN_NAME, isPrivate=True)
json.dump(meta, open(mp, "w"))
r = subprocess.run("kaggle models create -p " + meta_dir, shell=True)   # 'already exists' is fine
print("model create rc:", r.returncode)
subprocess.run("kaggle models instances init -p " + MERGED, shell=True, check=True)
ip = MERGED + "/model-instance-metadata.json"
im = json.load(open(ip))
im.update(ownerSlug=OWNER, modelSlug=RUN_NAME, instanceSlug="bf16", framework="transformers")
json.dump(im, open(ip, "w"))
r = subprocess.run("kaggle models instances create -p " + MERGED, shell=True)
if r.returncode:     # instance exists -> a new version
    subprocess.run("kaggle models instances versions create " + OWNER + "/" + RUN_NAME +
                   "/transformers/bf16 -p " + MERGED + " -n update", shell=True, check=True)


In [ ]:
# verify the uploaded copy, then hand over
import subprocess
subprocess.run("kaggle models instances versions download " + OWNER + "/" + RUN_NAME +
               "/transformers/bf16/1 -p /content/verify --untar", shell=True, check=True)
subprocess.run("cd /content/repo && python arc-agi-2/dev/check_swap_model.py /content/verify /content/base",
               shell=True, check=True)
print("DONE. On the ops VM: swap_runbook.md step 2 with "
      "--model-source " + OWNER + "/" + RUN_NAME + "/Transformers/bf16/1")


Done when the last cell prints DONE. The model source string printed there
goes straight into `make_gen_kernel.py --model-source` (or `make_union_kernel.py --second`)
for the panel comparison -- see `arc-agi-2/dev/swap_runbook.md`.
